# 文档助手 · Colab 一键启动

按顺序跑下面三格就行。**唯一必填的是硅基流动的 Key**。

1. 左侧工具栏 🔑（密钥）里加两个 Secret：
   - `SILICONFLOW_API_KEY` —— 必填，`sk-` 开头，[控制台](https://cloud.siliconflow.cn/account/ak)
   - `NGROK_AUTHTOKEN` —— 可选，填了自动开公网隧道；[免费领](https://dashboard.ngrok.com/get-started/your-authtoken)
2. 依次运行 ↓（运行时类型选不选 GPU 都行，选了扫描页识别快 3 倍左右）

> 没配 Secret 也能跑：第三格会弹框让你粘贴，或者直接回车跳过——那样只能解析文档和关键词检索，不能问答。

## 1 / 3　取代码

In [ ]:
REPO = "https://github.com/devinlin2022/RAG_project_Python.git"
DIR = "RAG_project_Python"

import os, pathlib, subprocess

root = pathlib.Path(DIR)
in_repo = (pathlib.Path.cwd() / "run.py").exists()
if not in_repo and not root.exists():
    subprocess.run(["git", "clone", "--quiet", REPO, DIR], check=True)
if root.exists() and not in_repo:
    os.chdir(root)
in_repo = (pathlib.Path.cwd() / "run.py").exists()

if in_repo and pathlib.Path(".git").exists():
    subprocess.run(["git", "pull", "--ff-only", "--quiet"], capture_output=True, text=True)

print("目录：", os.getcwd())
print("版本：", subprocess.run(["git", "log", "-1", "--oneline"],
                              capture_output=True, text=True).stdout.strip())

## 2 / 3　装依赖

大概 40 秒。末尾如果出现 `pip's dependency resolver` 的红色警告可以忽略——那是 Colab 自带的包被顶了版本，不影响本项目。

In [ ]:
!pip install -q -r requirements.txt
import gradio, uvicorn
print("依赖就绪：gradio", gradio.__version__, "｜ uvicorn", uvicorn.__version__)

## 3 / 3　启动

跑完会打印一个 `https://xxxx.ngrok-free.app` 的地址（没有 ngrok token 时是本地地址），点开就是聊天界面。

这一格**几秒就结束**，服务在后台线程里继续跑——不要以为它挂了。要停就 `from tools.colab import stop; stop()`。

In [ ]:
from tools.colab import boot
url = boot()

## 备用：没有 ngrok token 时

用 Colab 自带的端口代理开一个新标签页（能聊天，偶尔需要刷新一次）：

```python
from google.colab import output
output.serve_kernel_port_as_window(7860)
```

## 常见情况

| 现象 | 处理 |
|---|---|
| 改了 Key 想生效 | 直接重跑第 3 格，会自动把新 Key 同步进内存，不用重启会话 |
| 想换端口 / 重新起服务 | `from tools.colab import stop; stop()` 再跑第 3 格 |
| 提示端口已占用 | 说明上一次还在跑，第 3 格会直接把地址重打一遍，不重复启动 |
| 视觉识别显示"关闭" | 检查 Secret 里的 Key 是否有效；纯文字 PDF 本来就不会触发视觉，不产生费用 |